# Vision 2050 — Exploratory Analysis

Interactive exploration of policy extraction results and WDI trends.

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from config import COUNTRIES, WDI_INDICATORS, SDG_LABELS, PROCESSED_DIR, WDI_DIR, OUTPUTS_DIR

plt.style.use('dark_background')
pd.set_option('display.max_colwidth', 80)
print('Setup complete')

## 1. Load Comparison Table

In [ ]:
comparison_path = os.path.join(OUTPUTS_DIR, 'country_comparison.json')
if os.path.exists(comparison_path):
    df = pd.read_json(comparison_path)
    print(f'Loaded {len(df)} countries')
    display(df[['country_name','plan','composite_score','on_track','at_risk','off_track','sdg_count']].sort_values('composite_score', ascending=False))
else:
    print('No comparison table yet — run: python src/pipeline.py')

## 2. Inspect a Country's Extracted Profile

In [ ]:
COUNTRY = 'RWA'  # Change this

scored_path = os.path.join(PROCESSED_DIR, f'{COUNTRY}_scored.json')
if os.path.exists(scored_path):
    with open(scored_path) as f:
        profile = json.load(f)
    print(f"=== {profile['country_name']} — {profile['plan_name']} ===")
    print(f"Summary: {profile['plan_summary']}")
    print(f"\nPriority sectors: {', '.join(profile['priority_sectors'])}")
    print(f"SDGs covered: {sorted(profile['sdg_coverage'])}")
    print(f"Composite score: {profile['composite_score']}")
    print(f"\n--- TARGETS ---")
    for t in profile['scored_targets']:
        emoji = {'on_track':'✅','at_risk':'⚠️','off_track':'❌','no_data':'❓'}.get(t['track_status'],'?')
        ratio = f"{t['progress_ratio']:.0%}" if t['progress_ratio'] else 'N/A'
        print(f"{emoji} [{t['sector']}] {t['description'][:60]} — {ratio}")
else:
    print(f'No scored profile for {COUNTRY}. Run the pipeline first.')

## 3. WDI Trend Explorer

In [ ]:
INDICATOR = 'NY.GDP.PCAP.CD'  # Change this
COUNTRIES_TO_PLOT = ['RWA', 'KEN', 'GHA', 'ETH', 'TZA']  # Change this

series = []
for code in COUNTRIES_TO_PLOT:
    wdi_path = os.path.join(WDI_DIR, f'{code}_wdi.json')
    if not os.path.exists(wdi_path):
        continue
    with open(wdi_path) as f:
        data = json.load(f)
    df_c = pd.DataFrame(data['records'])
    df_c = df_c[df_c['indicator'] == INDICATOR][['year','value']]
    df_c['country'] = COUNTRIES[code]['name']
    series.append(df_c)

if series:
    combined = pd.concat(series)
    label = WDI_INDICATORS.get(INDICATOR, {}).get('label', INDICATOR)
    fig = px.line(combined, x='year', y='value', color='country',
                  title=f'{label} — Selected Countries',
                  template='plotly_dark')
    fig.show()
else:
    print('No WDI data loaded. Run: python src/wdi_fetch.py')

## 4. SDG Coverage Analysis

In [ ]:
# Count how many countries cover each SDG
sdg_counts = {}
for code in COUNTRIES:
    path = os.path.join(PROCESSED_DIR, f'{code}_extracted.json')
    if os.path.exists(path):
        with open(path) as f:
            p = json.load(f)
        for sdg in p.get('sdg_coverage', []):
            sdg_counts[sdg] = sdg_counts.get(sdg, 0) + 1

if sdg_counts:
    sdg_df = pd.DataFrame([
        {'SDG': f"SDG {k}", 'Goal': SDG_LABELS.get(k,''), 'Countries': v}
        for k, v in sorted(sdg_counts.items())
    ])
    fig = px.bar(sdg_df, x='SDG', y='Countries', hover_data=['Goal'],
                 title='Number of Countries Addressing Each SDG',
                 template='plotly_dark', color='Countries',
                 color_continuous_scale='Teal')
    fig.show()
else:
    print('No extracted profiles found. Run the pipeline first.')